# 분류를 위해 미세 튜닝하기 목차
* [Chapter 1 여러가지 미세 튜닝 방법](#chapter1)
* [Chapter 2 데이터셋 준비](#chapter2)
* [Chapter 3 데이터 로더 만들기](#chapter3)
* [Chapter 4 사전 훈련된 가중치로 모델 초기화하기](#chapter4)

## Chapter 1 여러가지 미세 튜닝 방법 <a class="anchor" id="chapter1"></a>
1. 언어 모델을 미세 튜닝하는 가장 일반적인 방법은 지시 미세 튜닝(instruction fine-tuning)과 분류 미세 튜닝(classification fine-tuning)입니다.
   - 지시 미세 튜닝은 구체적인 지시 데이터를 사용해 일련의 작업에서 언어 모델을 훈련한다.
      - 모델이 사용자의 구체적인 지시를 이해하고 이를 기반으로 응답을 생성하는 능력을 향상시킨다.
      - 복잡한 사용자의 지시를 기반으로 다양한 작업을 처리해야하는 모델에 잘 맞는다.
      - 다양한 작업에 능숙한 모델을 개발하려면 데이터셋과 컴퓨팅 자원이 많이 필요하다.
      - 예) 영어 문장을 독일어로 변경하라는 지시를 수행한다
   - 분류 미세 튜닝은 레이블이 있는 데이터셋을 사용해 모델이 특정 클래스에 대한 예측을 수행하도록 훈련한다.
      - 감성 분석이나 스팸 감지와같은 데이터를 사전에 정의된 클래스로 정확히 분류해야 하는 프로젝트에 적합하다
      - 데이터와 컴퓨팅 자원이 비교적 적게 필요하지만 모델이 훈련된 특정 클래스로만 사용이 제한된다.
      - 예) 모델이 텍스크가 스펨인지 아닌지 결정하는 작업을 수행한다.

      ![메세 튜닝](image/06-01-tunning2.png)

      ![메세 튜닝2](image/06-01-tunning4.png)



## Chapter 2 데이터셋 준비 <a class="anchor" id="chapter2"></a>
1. 이전에 구현하고 사전 훈련한 GPT 모델을 수정해서 분류 미세 튜닝을 수행할 수 있다.

2. 분류 미세 튜닝에 유용한 예제로 '스펨'과 '스팸 아님'으로 구성된 텍스트 메시지 데이터셋을 사용한다.
   - 텍스트 메시지는 일반적으 이메일이 아니라 핸드폰으로 전달되지만, 동일한 단계가 이메일 분류에도 적용된다.

   ![프로세스](image/06-01-process2.png)

In [2]:
# 데이터셋 다운로드
import urllib.request
import zipfile
import os
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"

def download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():
        print(f"{data_file_path}가 이미 있어 다운로드 및 압축 해제를 건너뜁니다.")
        return

    with urllib.request.urlopen(url) as response:
        with open(zip_path, 'wb') as out_file:
            out_file.write(response.read())

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extracted_path)
   
    # .tsv 파일 확장자를 추가합니다.
    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"파일이 다운로드되어 {data_file_path}에 저장되었습니다.")
    
download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)


파일이 다운로드되어 sms_spam_collection/SMSSpamCollection.tsv에 저장되었습니다.


In [4]:
# 판다스 데이터프레임으로 로드
import pandas as pd
df = pd.read_csv(data_file_path, sep='\t', header=None, names=['Label', 'Text'])
print(df)

     Label                                               Text
0      ham  Go until jurong point, crazy.. Available only ...
1      ham                      Ok lar... Joking wif u oni...
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...
3      ham  U dun say so early hor... U c already then say...
4      ham  Nah I don't think he goes to usf, he lives aro...
...    ...                                                ...
5567  spam  This is the 2nd time we have tried 2 contact u...
5568   ham               Will ü b going to esplanade fr home?
5569   ham  Pity, * was in mood for that. So...any other s...
5570   ham  The guy did some bitching but I acted like i'd...
5571   ham                         Rofl. Its true to its name

[5572 rows x 2 columns]


In [5]:
# 레이블 분포 조사
#   - 레이블이 'ham'인 메시지와 'spam'인 메시지의 개수를 출력
print(df['Label'].value_counts())

Label
ham     4825
spam     747
Name: count, dtype: int64


In [ ]:
# 미세퓨닝을 빠르게 진행하기 위해 747개의 샘플만 포함되도록 데이터셋 줄이기
def create_balanced_dataset(df):

    # "스팸" 샘플 개수 세기
    num_spam = df[df["Label"] == "spam"].shape[0]

    # "스팸" 샘플 개수와 일치하도록 "햄" 샘플을 무작위로 샘플링
    ham_subset = df[df["Label"] == "ham"].sample(num_spam, random_state=123)

    # "햄"과 "스팸"을 합침
    balanced_df = pd.concat([ham_subset, df[df["Label"] == "spam"]])

    return balanced_df


balanced_df = create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())

Label
ham     747
spam    747
Name: count, dtype: int64


In [ ]:
# 문자열로 된 클래스 레이블 "ham"과 "spam"을 정수로 변환
#   - 텍스트를 토큰 ID로 변환하는 것과 유사하다.
#   - 50,000개 단어 이상으로 구성된 GPT 어휘 사전을 사용하지 않고 0과 1로 레이블을 인코딩
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})
balanced_df

,Label,Text
4307,0,Awww dat is sweet! We can think of something t...
4138,0,Just got to &lt;#&gt;
4831,0,"The word ""Checkmate"" in chess comes from the P..."
4461,0,This is wishing you a great day. Moji told me ...
5440,0,Thank you. do you generally date the brothas?
...,...,...
5537,1,Want explicit SEX in 30 secs? Ring 02073162414...
5540,1,ASKED 3MOBILE IF 0870 CHATLINES INCLU IN FREE ...
5547,1,Had your contract mobile 11 Mnths? Latest Moto...
5566,1,REMINDER FROM O2: To get 2.50 pounds free call...


In [13]:
# 데이터셋을 세 부분으로 분활하는 random_split 함수 정의
#   - 훈련 세트(70%), 검증 세트(10%), 테스트 세트(20%)
def random_split(df, train_frac, validation_frac):
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)  # 데이터프레임을 무작위로 섞기
    train_end = int(len(df) * train_frac) # 훈련 세트 분할할 인덱스 계산
    validation_end = train_end + int(len(df) * validation_frac) # 검증 세트 분할할 인덱스 계산
    
    train_df = df[:train_end] # 훈련 세트
    validation_df = df[train_end:validation_end] # 검증 세트
    test_df = df[validation_end:] # 테스트 세트

    return train_df, validation_df, test_df

train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)

In [14]:
# 데이터셋을 나중에 사용하기 위해 CSV 파일로 저장
train_df.to_csv("train.csv", index=False)
validation_df.to_csv("validation.csv", index=False)
test_df.to_csv("test.csv", index=False)

## Chapter 3 데이터 로더 만들기 <a class="anchor" id="chapter3"></a>
1. 데이터셋 길이 정리 방법
   - 가장 짧은 길이의 메시지에 맞춰 모든 메시지를 잘라낸다.
        - 계산 비용이 저렴하지만 정보 손실이 발생할 수 있다.
   - 가장 긴 길의 메시지에 맞춰 모든 메시지에 패딩을 추가한다.
        - 정보 손실이 없지만 계산 비용이 증가한다.

2. 가장 긴 메시지의 길이로 모든 메시지를 맞추어 배치를 생성한다.
    - 짧은 길이의 메시지에 패딩 토큰을 추가한다.
    - "<|endoftext|>" 토큰에 해당하는 토큰 ID(50256)로 패딩을 수행한다.

In [15]:
# 토큰 ID 50256이 "<|endoftext|>" 토큰에 해당하는지 확인
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.decode([50256]))  # 출력: <|endoftext|>

<|endoftext|>


3. 데이터 로더를 만들기 전에 데이터를 로드하고 처리하는 방법을 지정하기 위해 파이토치 데이터셋 클래스를 구현한다.
    - SpamDataset 클래스를 정의한다.
        - 텍스트 메시지를 토큰ID 시퀸스로 인코딩
        - 훈련 데이터셋에서 가장 긴 시퀸스를 식별
        - 모든 시퀸스에 패딩토큰을 추가하여 가장 긴 시퀸스 길이에 맞춤  

        ![패딩](image/06-01-padding2.png)

In [30]:
import torch
from torch.utils.data import Dataset


class SpamDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=None, pad_token_id=50256):
        self.data = pd.read_csv(csv_file)

        # 텍스트 토큰화
        self.encoded_texts = [
            tokenizer.encode(text) for text in self.data["Text"]
        ]

        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length
            # max_length보다 긴 시퀀스 자르기
            self.encoded_texts = [
                encoded_text[:self.max_length]
                for encoded_text in self.encoded_texts
            ]

        # 가장 긴 시퀀스에 맞춰 패딩하기
        self.encoded_texts = [
            encoded_text + [pad_token_id] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["Label"]
        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )

    def __len__(self):
        return len(self.data)

    def _longest_encoded_length(self):
        max_length = 0
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length > max_length:
                max_length = encoded_length
        return max_length
        # 참고: 이 메서드를 구현하는 더 파이썬적인 버전은
        # 다음과 같으며, 다음 장에서도 사용됩니다.
        # return max(len(encoded_text) for encoded_text in self.encoded_texts)

In [31]:
train_dataset = SpamDataset(csv_file="train.csv", max_length=None, tokenizer=tokenizer)

print(train_dataset.max_length)

120


In [32]:
val_dataset = SpamDataset(csv_file="validation.csv", max_length=train_dataset.max_length, tokenizer=tokenizer)
print(val_dataset.max_length)

120


In [33]:
test_dataset = SpamDataset(csv_file="test.csv", max_length=train_dataset.max_length, tokenizer=tokenizer)
print(test_dataset.max_length)

120


In [ ]:
# 연습문제 6.1: 문맥 길이 늘리기
# 모델이 지원하는 최대 토큰 수로 입력을 패딩하려면 max_length를 1024로 설정합니다.
max_length = 1024

train_dataset = SpamDataset(csv_file="train.csv", max_length=max_length, tokenizer=tokenizer)
val_dataset = SpamDataset(csv_file="validation.csv", max_length=max_length, tokenizer=tokenizer)
test_dataset = SpamDataset(csv_file="test.csv", max_length=max_length, tokenizer=tokenizer)

4. 배치 크기 8, 배치 길이 120인 훈련 샘플과 각 샘플에 해당하는 클래스레이블로 데이터 로더 구성
    - 텍스트 메시지와 레이블을 로드하여 배치 크기 8인 훈련, 검증, 테스트 세트 데이터 로더를 만든다.

    ![데이터로더](image/06-01-batch2.png)

In [34]:
from torch.utils.data import DataLoader

num_workers = 0 # 데이터 로더에 사용할 CPU 코어 수
batch_size = 8 # 배치 크기
torch.manual_seed(123)  # 재현성을 위해 시드 설정

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True,
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

In [35]:
# 마지막 배치 텐서 차원 출력
print("훈련 세트 로더:")
for input_batch, target_batch in train_loader:
    pass

print("입력 배치 차원:", input_batch.shape)
print("레이블 배치 차원", target_batch.shape)

훈련 세트 로더:
입력 배치 차원: torch.Size([8, 120])
레이블 배치 차원 torch.Size([8])


In [36]:
print(f"{len(train_loader)}개 훈련 배치")
print(f"{len(val_loader)}개 검증 배치")
print(f"{len(test_loader)}개 테스트 배치")

130개 훈련 배치
19개 검증 배치
38개 테스트 배치


## Chapter 4 사전 훈련된 가중치로 모델 초기화하기 <a class="anchor" id="chapter4"></a>
1. 스팸 메시지를 식별하도록 분류 미세 튜닝히가 위한 모델 준비.
    - 이전에 사전 훈련한 GPT 모델을 로드한다.
    - 출력 레이어를 이진 분류를 수행하도록 수정한다.
    - 모델의 나머지 부분은 사전 훈련된 가중치로 초기화한다.

In [41]:
CHOOSE_MODEL = "gpt2-small (124M)"
INPUT_PROMPT = "Every effort moves"

BASE_CONFIG = {
    "vocab_size": 50257,     # 어휘사전 크기
    "context_length": 1024,  # 문맥 길이
    "drop_rate": 0.0,        # 드롭아웃 비율
    "qkv_bias": True         # 쿼리-키-값 편향
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

assert train_dataset.max_length <= BASE_CONFIG["context_length"], (
    f"데이터셋 길이 {train_dataset.max_length}가 모델의 문맥 "
    f"길이 {BASE_CONFIG['context_length']}를 초과합니다. `max_length={BASE_CONFIG['context_length']}`로 "
    f"데이터 셋을 다시 초기화하십시오."
)
 

In [38]:
!wget https://bit.ly/4jZL2Gr -O gpt_download.py
!wget https://bit.ly/4esl8dj -O previous_chapters.py

--2025-10-22 16:18:15--  https://bit.ly/4jZL2Gr
Resolving bit.ly (bit.ly)... 67.199.248.11, 67.199.248.10
Connecting to bit.ly (bit.ly)|67.199.248.11|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://raw.githubusercontent.com/rickiepark/llm-from-scratch/refs/heads/main/ch06/01_main-chapter-code/gpt_download.py [following]
--2025-10-22 16:18:15--  https://raw.githubusercontent.com/rickiepark/llm-from-scratch/refs/heads/main/ch06/01_main-chapter-code/gpt_download.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6333 (6.2K) [text/plain]
Saving to: ‘gpt_download.py’

gpt_download.py     100%[===================>]   6.18K  --.-KB/s    in 0s      

2025-10-22 16:18:15 (60.6 MB/s) - ‘gpt_download.py’ saved [6333/63

In [43]:
# 기존 작성된 파일을 로드하여 GTP 모델로 로드
from gpt_download import download_and_load_gpt2
from previous_chapters import GPTModel, load_weights_into_gpt

model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")  # "gpt2-small"
print(f"모델 크기: {model_size}")

settings, params = download_and_load_gpt2(model_size=model_size, models_dir="gpt2")

model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
print(model)

model.eval()  # 평가 모드로 전환

모델 크기: 124M
File already exists and is up-to-date: gpt2/124M/checkpoint
File already exists and is up-to-date: gpt2/124M/encoder.json
File already exists and is up-to-date: gpt2/124M/hparams.json
File already exists and is up-to-date: gpt2/124M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2/124M/model.ckpt.index
File already exists and is up-to-date: gpt2/124M/model.ckpt.meta
File already exists and is up-to-date: gpt2/124M/vocab.bpe
GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, in

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_resid): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768,

In [44]:
# 텍스트 생성 유틸리티를 사용해 모델이 일관적인 텍스트를 생성하는지 확인
from previous_chapters import generate_text_simple
from previous_chapters import text_to_token_ids, token_ids_to_text

text_1 = "Every effort moves you"
token_ids = generate_text_simple(
    model,
    text_to_token_ids(text_1, tokenizer),
    max_new_tokens=15,
    context_size=BASE_CONFIG["context_length"]
)
print(token_ids_to_text(token_ids, tokenizer))

Every effort moves you forward.

The first step is to understand the importance of your work


In [ ]:
# 모델을 스팸 분류기로 미세튜닝하기 전에 명령이 포함된 프롬프로 스팸을 분류 할 수 있는지 확인
text_2 = (
    "Is the following text 'spam'? Answer with 'yes' or 'no':"
    " 'You are a winner you have been specially"
    " selected to receive $1000 cash or a $2000 award.'"
)
token_ids = generate_text_simple(
    model,
    text_to_token_ids(text_2, tokenizer),
    max_new_tokens=10,
    context_size=BASE_CONFIG["context_length"]
)
print(token_ids_to_text(token_ids, tokenizer))